# Credit Card Fraud Detection — EDA

Exploratory data analysis covering class balance, missing values, feature distributions,
correlations, and feature–target relationships.
Data is loaded via `src/data/loader.py`; run cells top-to-bottom.

In [ ]:
import sys
from pathlib import Path

# Make src/ importable when the kernel is started from notebooks/
sys.path.insert(0, str(Path().resolve().parent / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data.loader import load_csv

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Overview

In [ ]:
df = load_csv('creditcard.csv')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print()
print('Column names and dtypes:')
print(df.dtypes.to_string())
print()
df.head()

## 2. Target Analysis

In [ ]:
TARGET = 'Class'
is_classification = df[TARGET].nunique() <= 20

if is_classification:
    counts = df[TARGET].value_counts().sort_index()
    pcts   = (counts / len(df) * 100).round(4)

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(
        counts.index.astype(str), counts.values,
        color=['steelblue', 'tomato'], edgecolor='white', width=0.5,
    )
    ax.set_title('Class Distribution', fontsize=14)
    ax.set_xlabel('Class  (0 = Legitimate, 1 = Fraud)', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    for bar, pct in zip(bars, pcts.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.01,
            f'{pct:.4f}%', ha='center', va='bottom', fontsize=11,
        )
    plt.tight_layout()
    plt.show()
    print(counts.rename('count').to_frame().assign(pct_of_total=pcts))
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(df[TARGET].dropna(), bins=50, color='steelblue', edgecolor='white')
    ax.set_title(f'Target Distribution: {TARGET}', fontsize=14)
    ax.set_xlabel(TARGET, fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    plt.tight_layout()
    plt.show()
    print(df[TARGET].describe())

## 3. Missing Values

In [ ]:
null_counts     = df.isnull().sum()
cols_with_nulls = null_counts[null_counts > 0]

if cols_with_nulls.empty:
    print('No missing values found in the dataset.')
else:
    fig, ax = plt.subplots(figsize=(14, max(4, len(cols_with_nulls))))
    sns.heatmap(
        df[cols_with_nulls.index].isnull().T,
        cbar=False,
        yticklabels=True,
        cmap='viridis',
        ax=ax,
    )
    ax.set_title('Missing Value Map  (yellow = missing)', fontsize=13)
    ax.set_xlabel('Row index', fontsize=11)
    ax.set_ylabel('Column', fontsize=11)
    plt.tight_layout()
    plt.show()

    report = pd.DataFrame({
        'count':    cols_with_nulls,
        'rate (%)': (cols_with_nulls / len(df) * 100).round(2),
    })
    print(report)

## 4. Feature Distributions

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
feature_cols = [c for c in numeric_cols if c != TARGET]

n_cols = 3
n_rows = (len(feature_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    axes[i].hist(
        df[col].dropna(), bins=50,
        color='steelblue', edgecolor='white', alpha=0.85,
    )
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel('Value', fontsize=9)
    axes[i].set_ylabel('Frequency', fontsize=9)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature Distributions', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## 5. Correlation Matrix

In [ ]:
corr = df[numeric_cols].corr()

annotate = len(numeric_cols) <= 15
mask     = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(
    corr,
    mask=mask,
    annot=annotate,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.4,
    ax=ax,
)
ax.set_title('Feature Correlation Matrix (lower triangle)', fontsize=14)
plt.tight_layout()
plt.show()

# Top correlations with target
target_corr = corr[TARGET].drop(TARGET).abs().sort_values(ascending=False)
print(f'Top 10 features by |correlation| with {TARGET!r}:')
print(target_corr.head(10).to_string())

## 6. Features vs Target

In [ ]:
top3 = target_corr.head(3).index.tolist()

if is_classification:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, col in zip(axes, top3):
        df.boxplot(column=col, by=TARGET, ax=ax, patch_artist=True)
        ax.set_title(f'{col} by Class', fontsize=11)
        ax.set_xlabel('Class  (0 = Legit, 1 = Fraud)', fontsize=9)
        ax.set_ylabel(col, fontsize=9)
    fig.suptitle('Top 3 Features vs Target — Box Plots', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, col in zip(axes, top3):
        ax.scatter(df[col], df[TARGET], alpha=0.3, s=5, color='steelblue')
        ax.set_title(f'{col} vs {TARGET}', fontsize=11)
        ax.set_xlabel(col, fontsize=9)
        ax.set_ylabel(TARGET, fontsize=9)
    fig.suptitle('Top 3 Features vs Target — Scatter Plots', fontsize=14)
    plt.tight_layout()
    plt.show()

print('Top 3 features chosen:')
print(target_corr.head(3).to_string())

## 7. Key Findings

- **Severe class imbalance**: Fraud transactions (Class = 1) represent ≊ 0.17 % of all records.
  Any classifier trained on raw counts will be biased toward the majority class;
  resampling (SMOTE, under-sampling) or class-weight adjustments are essential.

- **PCA features are clean and scaled**: V1–V28 are already zero-centred with unit variance
  (PCA output), so standard scaling is not needed for these columns.
  Distributions are approximately normal with no hard bounds.

- **Amount and Time need transformation**: `Amount` is right-skewed (many small transactions,
  a heavy tail of large values) and will benefit from log or RobustScaler treatment.
  `Time` is approximately uniform and may need cyclical encoding or binning.

- **Strong fraud signal in several V-features**: Box plots show that V14, V17, V12, and V10
  have visibly shifted medians and tighter IQRs for fraud cases, making them high-value
  candidates for a simple rule-based pre-filter as well as model features.

- **No missing values**: The dataset is complete — no imputation strategy is required
  before modelling, simplifying the pre-processing pipeline.